# Part B: vLLM on a Colab T4 GPU

This notebook serves **Qwen2.5-3B-Instruct** with vLLM in two versions, **FP16** and **AWQ 4-bit**,
and runs the same latency and quality benchmarks as Part A.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells **from top to bottom, one at a time**. Wait for each cell to finish
(the spinner stops) before running the next.

## Step 1: Check the GPU
You should see `Tesla T4` and about 15 GB of memory. If not, change the runtime type (see above).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## Step 2: Get the code from GitHub
Change `REPO_URL` if your repo has a different name. If the folder already exists
(for example after a reconnect), this pulls the latest version instead of cloning again.

In [ ]:
import os

REPO_URL = "https://github.com/mandalika-sravani/local-llm-bench.git"
WORKDIR = "/content/bench"

if os.path.exists(WORKDIR):
    !git -C {WORKDIR} pull
else:
    !git clone {REPO_URL} {WORKDIR}

%cd {WORKDIR}
!ls

## Step 3: Install vLLM and the benchmark requirements
Takes a few minutes. pip may print **"ERROR: pip's dependency resolver..."** about packages
like `google-adk`, `cudf` or `numba`. That warning is about Colab's preinstalled packages and can be ignored.
The last line should print a vLLM version number.

In [ ]:
!pip install -q vllm -r requirements.txt
!python -c "import vllm; print('vLLM version:', vllm.__version__)"

## Step 4: Helper functions (run once)
- `start_vllm(...)` stops any running server, waits for the GPU to free its memory, starts a new server,
  and **waits until it is ready**. If the server crashes, it prints the end of the log.
- `vllm_memory(...)` prints the real weight and KV cache sizes from the server log.
- `count_answers()` checks that each answer file has all 20 answers.

If Colab disconnects, rerun Steps 2 and 4 (and Step 3 if the import fails).

In [ ]:
import glob, subprocess, time
import requests

SERVER_URL = "http://localhost:8000/v1/models"
server = None  # the running vLLM process


def gpu_memory_used_mb():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    ).stdout
    return int(out.strip().splitlines()[0])


def stop_vllm():
    global server
    subprocess.run("pkill -f 'vllm serve'", shell=True)
    server = None
    for _ in range(30):  # wait up to 60 s for the GPU to be freed
        if gpu_memory_used_mb() < 1000:
            break
        time.sleep(2)
    print(f"Server stopped. GPU memory in use: {gpu_memory_used_mb()} MB")


def start_vllm(model, log_file, extra_args=()):
    global server
    stop_vllm()
    cmd = ["vllm", "serve", model,
           "--max-model-len", "4096",
           "--gpu-memory-utilization", "0.85", *extra_args]
    print("Starting:", " ".join(cmd))
    server = subprocess.Popen(cmd, stdout=open(log_file, "w"), stderr=subprocess.STDOUT)

    for waited in range(0, 900, 5):  # wait up to 15 minutes (first run downloads the model)
        if server.poll() is not None:
            raise RuntimeError("vLLM crashed. End of log:\n" + open(log_file).read()[-3000:])
        try:
            if requests.get(SERVER_URL, timeout=2).ok:
                print(f"vLLM is ready after {waited} s: {model}")
                return
        except requests.exceptions.RequestException:
            pass
        if waited % 30 == 0:
            print(f"  still loading... {waited} s")
        time.sleep(5)
    raise RuntimeError(f"vLLM did not become ready in 15 minutes. Check {log_file}.")


def vllm_memory(log_file):
    print(f"--- {log_file} ---")
    for line in open(log_file):
        low = line.lower()
        if "weights took" in low or "kv cache" in low or "gpu blocks" in low:
            print(line.strip()[-160:])


def count_answers():
    for path in sorted(glob.glob("results/answers_*.jsonl")):
        n = sum(1 for line in open(path, encoding="utf-8") if line.strip())
        status = "OK" if n == 20 else "INCOMPLETE, rerun this model"
        print(f"{n:>3} answers  {path}  {status}")


print("Helpers loaded.")

---
# FP16 model

## Step 5: Start the FP16 server
T4 GPUs don't support BF16, so `--dtype half` (FP16) is required. The first start downloads about 6 GB.

In [ ]:
FP16_MODEL = "Qwen/Qwen2.5-3B-Instruct"
start_vllm(FP16_MODEL, "vllm_fp16.log", ["--dtype", "half"])
vllm_memory("vllm_fp16.log")

## Step 6: FP16 latency benchmark
**Skip this cell if you already have the FP16 rows in `results/latency.csv`**, because running it again adds duplicate rows.

In [ ]:
!python -m bench.latency --backend vllm --model {FP16_MODEL} --label fp16 --concurrency 1,4,8,16 --runs 16 --max-tokens 200

## Step 7: FP16 quality answers
Generates answers to the 20 prompts (about a minute on the GPU).

In [ ]:
!python -m bench.quality generate --backend vllm --model {FP16_MODEL} --max-tokens 250
count_answers()

---
# AWQ 4-bit model

## Step 8: Start the AWQ server
This stops the FP16 server first. AWQ weights are about 2 GB.

In [ ]:
AWQ_MODEL = "Qwen/Qwen2.5-3B-Instruct-AWQ"
start_vllm(AWQ_MODEL, "vllm_awq.log")
vllm_memory("vllm_awq.log")

## Step 9: AWQ latency benchmark
**Skip this cell if you already have the AWQ rows in `results/latency.csv`.**

In [ ]:
!python -m bench.latency --backend vllm --model {AWQ_MODEL} --label awq --concurrency 1,4,8,16 --runs 16 --max-tokens 200

## Step 10: AWQ quality answers

In [ ]:
!python -m bench.quality generate --backend vllm --model {AWQ_MODEL} --max-tokens 250
count_answers()

---
# Finish

## Step 11: Check results, stop the server, download
Both `answers_vllm_*` files should say **20 answers OK**. Then download `results.zip`,
unzip it into your project folder on your laptop, and run the judge there:
`python -m bench.quality judge "results/answers_*.jsonl"`

In [ ]:
count_answers()
vllm_memory("vllm_fp16.log")
vllm_memory("vllm_awq.log")
stop_vllm()

!zip -qr results.zip results vllm_fp16.log vllm_awq.log
from google.colab import files
files.download("results.zip")